# 第13章（三）：浮点数表示与运算 Floating Point Representation

## Cambridge A-Level Computer Science (9618)

---

### 本节学习目标 Learning Objectives

1. 理解浮点数表示法：尾数 (mantissa) 与阶码 (exponent)
2. 掌握二进制浮点数的转换方法
3. 理解规格化 (normalisation) 及其必要性
4. 理解精度 (precision)、范围 (range)、上溢 (overflow) 和下溢 (underflow)
5. 掌握浮点数加法和减法运算
6. 深入理解为什么 0.1 + 0.2 != 0.3

---

### 引言：为什么需要浮点数？

整数 (integer) 只能表示 ...,-2,-1,0,1,2,...

但现实世界中我们需要表示：
- 圆周率 pi = 3.14159...
- 光速 c = 3.0 x 10^8 m/s
- 电子质量 = 9.109 x 10^-31 kg

**浮点数** 的核心思想来自**科学记数法 (scientific notation)**：

> 3.14159 = **3.14159** x 10^**0**  
> 300000000 = **3.0** x 10^**8**  
> 0.0000000000000000000000000000009109 = **9.109** x 10^**-31**

在二进制中，同样的思路：

> value = **mantissa** x 2^**exponent**

In [ ]:
# 环境配置
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import numpy as np
import struct

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

print('环境配置完成！')

---

## 一、浮点数表示法 Floating Point Representation

### 核心公式

$$\text{value} = \text{mantissa} \times 2^{\text{exponent}}$$

- **尾数 Mantissa**：表示数的**有效数字**（决定精度 precision）
- **阶码 Exponent**：表示小数点的**移动位数**（决定范围 range）

### 在Cambridge考试中的表示

通常给定总位数（如8位），分为两部分：
- 前面若干位是**尾数**（使用**二进制补码 two's complement**）
- 后面若干位是**阶码**（使用**二进制补码**）

例如：8位浮点数 = 5位尾数 + 3位阶码

```
| m m m m m | e e e |
| 尾数(5位)  | 阶码(3位)|
```

### 为什么用二进制补码？

因为需要表示**正数和负数**。二进制补码的最高位是**符号位**（0=正，1=负）。

In [ ]:
# === 浮点数表示法基础演示 ===

def twos_complement_value(bits):
    """计算二进制补码的十进制值"""
    n = len(bits)
    if bits[0] == 1:  # 负数
        # 符号位的权重是 -2^(n-1)
        value = -2**(n-1)
        for i in range(1, n):
            value += bits[i] * 2**(n-1-i)
        return value
    else:  # 正数
        value = 0
        for i in range(n):
            value += bits[i] * 2**(n-1-i)
        return value

def mantissa_value(bits):
    """计算尾数的十进制值（小数）
    尾数被视为纯小数：位权为 -1, 2^-1, 2^-2, ...
    最高位是符号位（权重 -2^0 = -1）
    """
    n = len(bits)
    if bits[0] == 1:  # 负数
        value = -1  # 符号位权重 = -1 (即 -2^0)
    else:
        value = 0
    for i in range(1, n):
        value += bits[i] * (2 ** (-i))
    return value

def float_to_decimal(mantissa_bits, exponent_bits):
    """将浮点数位模式转换为十进制"""
    m = mantissa_value(mantissa_bits)
    e = twos_complement_value(exponent_bits)
    result = m * (2 ** e)
    return m, e, result

# === 示例演示 ===
print("=== 浮点数转换演示 Float Conversion Demo ===")
print("格式：5位尾数 + 3位阶码\n")

examples = [
    ([0, 1, 1, 0, 0], [0, 1, 0], "正数示例"),
    ([0, 1, 0, 0, 0], [1, 0, 0], "正数小值"),
    ([1, 0, 1, 0, 0], [0, 1, 1], "负数示例"),
    ([0, 1, 1, 1, 0], [0, 0, 0], "阶码为0"),
]

for mantissa, exponent, label in examples:
    m, e, result = float_to_decimal(mantissa, exponent)
    m_str = ''.join(map(str, mantissa))
    e_str = ''.join(map(str, exponent))
    print(f"{label}:")
    print(f"  位模式: [{m_str} | {e_str}]")
    print(f"  尾数 mantissa = {m} (二进制小数)")
    print(f"  阶码 exponent = {e}")
    print(f"  值 value = {m} x 2^{e} = {result}")
    print()

In [ ]:
# === 详细的转换过程演示 ===

print("=== 逐步转换过程 Step-by-Step Conversion ===")
print()
print("例题：将 01100 | 010 转换为十进制")
print()
print("Step 1: 计算尾数（5位二进制补码小数）")
print("  位模式:    0    1    1    0    0")
print("  位权:    -2^0  2^-1 2^-2 2^-3 2^-4")
print("  位权值:   -1   0.5  0.25 0.125 0.0625")
print("  贡献:     0  + 0.5 + 0.25 + 0  + 0 = 0.75")
print()
print("Step 2: 计算阶码（3位二进制补码整数）")
print("  位模式: 0 1 0")
print("  = 0*(-4) + 1*2 + 0*1 = 2")
print()
print("Step 3: 计算最终值")
print("  value = mantissa x 2^exponent")
print("  value = 0.75 x 2^2")
print("  value = 0.75 x 4")
print("  value = 3.0")
print()

# 验证
m, e, result = float_to_decimal([0,1,1,0,0], [0,1,0])
print(f"验证：{m} x 2^{e} = {result} ✓")

In [ ]:
# === 可视化：浮点数的位结构 ===

fig, ax = plt.subplots(figsize=(14, 5))

# 绘制位格式
mantissa_bits = [0, 1, 1, 0, 0]
exponent_bits = [0, 1, 0]

total_bits = len(mantissa_bits) + len(exponent_bits)
cell_width = 1.2
cell_height = 1.0
start_x = (14 - total_bits * cell_width) / 2

# 尾数位
for i, bit in enumerate(mantissa_bits):
    x = start_x + i * cell_width
    color = '#e74c3c' if i == 0 else '#3498db'  # 符号位用红色
    rect = mpatches.FancyBboxPatch((x, 2), cell_width - 0.1, cell_height,
                                     boxstyle="round,pad=0.05",
                                     facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + cell_width/2 - 0.05, 2.5, str(bit), ha='center', va='center',
            fontsize=20, fontweight='bold', color='white')
    # 位权标签
    if i == 0:
        weight = '-1'
    else:
        weight = f'2^-{i}'
    ax.text(x + cell_width/2 - 0.05, 1.5, weight, ha='center', va='center',
            fontsize=10, color='gray')

# 阶码位
for i, bit in enumerate(exponent_bits):
    x = start_x + (len(mantissa_bits) + i) * cell_width
    color = '#f39c12' if i == 0 else '#e67e22'
    rect = mpatches.FancyBboxPatch((x, 2), cell_width - 0.1, cell_height,
                                     boxstyle="round,pad=0.05",
                                     facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + cell_width/2 - 0.05, 2.5, str(bit), ha='center', va='center',
            fontsize=20, fontweight='bold', color='white')
    # 位权
    n_exp = len(exponent_bits)
    if i == 0:
        weight = f'-{2**(n_exp-1)}'
    else:
        weight = f'{2**(n_exp-1-i)}'
    ax.text(x + cell_width/2 - 0.05, 1.5, weight, ha='center', va='center',
            fontsize=10, color='gray')

# 括号标注
m_start = start_x
m_end = start_x + len(mantissa_bits) * cell_width - 0.1
e_start = start_x + len(mantissa_bits) * cell_width
e_end = start_x + total_bits * cell_width - 0.1

ax.annotate('', xy=(m_start, 3.3), xytext=(m_end, 3.3),
            arrowprops=dict(arrowstyle='<->', color='#3498db', lw=2))
ax.text((m_start + m_end) / 2, 3.6, 'Mantissa (5 bits)', ha='center', fontsize=12, 
        fontweight='bold', color='#3498db')

ax.annotate('', xy=(e_start, 3.3), xytext=(e_end, 3.3),
            arrowprops=dict(arrowstyle='<->', color='#e67e22', lw=2))
ax.text((e_start + e_end) / 2, 3.6, 'Exponent (3 bits)', ha='center', fontsize=12,
        fontweight='bold', color='#e67e22')

# 符号位标注
ax.annotate('Sign bit\n(\u7b26\u53f7\u4f4d)', xy=(start_x + cell_width/2 - 0.05, 2),
            xytext=(start_x + cell_width/2 - 0.05, 0.8),
            fontsize=10, color='#e74c3c', fontweight='bold', ha='center',
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5))

# 计算结果
ax.text(7, 4.3, 'value = 0.75 x 2^2 = 3.0', ha='center', fontsize=16, 
        fontweight='bold', color='#2c3e50',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#ecf0f1', edgecolor='#2c3e50'))

ax.set_xlim(0, 14)
ax.set_ylim(0.3, 5)
ax.axis('off')
ax.set_title('8-bit Floating Point: 5-bit Mantissa + 3-bit Exponent\n8\u4f4d\u6d6e\u70b9\u6570\u7ed3\u6784\u56fe', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 二、规格化 Normalisation

### 是什么？

规格化是确保浮点数**以最高精度存储**的过程。

### 为什么需要规格化？

同一个数可以有多种浮点表示方式：
- 0.5 = 0.1000 x 2^0 = 0.0100 x 2^1 = 0.0010 x 2^2

非规格化的表示**浪费了尾数的有效位**，降低了精度。

### 规格化规则

对于**二进制补码**尾数：

| 数的类型 | 规格化条件 | 尾数开头 | 含义 |
|:---|:---|:---|:---|
| **正数** | 尾数以 **0.1** 开头 | 0.1xxxx | 符号位0，最高有效位1 |
| **负数** | 尾数以 **1.0** 开头 | 1.0xxxx | 符号位1，最高有效位0 |

**关键理解**：规格化确保尾数的**绝对值**尽可能大（在0.5到1之间），从而**最大化有效位数**。

In [ ]:
# === 规格化演示 Normalisation Demo ===

def is_normalised(mantissa_bits):
    """检查尾数是否已规格化"""
    if len(mantissa_bits) < 2:
        return False
    # 正数：0.1... (前两位是 0,1)
    # 负数：1.0... (前两位是 1,0)
    return mantissa_bits[0] != mantissa_bits[1]

def normalise(mantissa_bits, exponent_bits):
    """规格化浮点数"""
    mantissa = list(mantissa_bits)
    exponent = twos_complement_value(exponent_bits)
    steps = []
    
    while len(mantissa) >= 2 and mantissa[0] == mantissa[1]:
        old_m = ''.join(map(str, mantissa))
        # 左移尾数（去掉第二位，末尾补位）
        removed_bit = mantissa[1]
        mantissa.pop(1)
        # 正数末尾补0，负数末尾补1
        mantissa.append(mantissa[0])  # 简化处理
        exponent -= 1
        new_m = ''.join(map(str, mantissa))
        steps.append(f"  Shift left: {old_m} -> {new_m}, exponent: {exponent+1} -> {exponent}")
    
    return mantissa, exponent, steps

print("=== 规格化演示 Normalisation Demo ===")
print()

test_cases = [
    ([0, 0, 1, 1, 0], [0, 1, 1], "正数未规格化"),
    ([0, 1, 1, 0, 0], [0, 1, 0], "正数已规格化"),
    ([1, 1, 0, 1, 0], [0, 1, 0], "负数未规格化"),
    ([1, 0, 1, 0, 0], [0, 1, 1], "负数已规格化"),
]

for mantissa, exponent, label in test_cases:
    m_str = ''.join(map(str, mantissa))
    e_str = ''.join(map(str, exponent))
    normalised = is_normalised(mantissa)
    status = "NORMALISED" if normalised else "NOT normalised"
    
    m_val = mantissa_value(mantissa)
    e_val = twos_complement_value(exponent)
    result = m_val * (2 ** e_val)
    
    print(f"{label}: [{m_str} | {e_str}]  ->  {status}")
    print(f"  value = {m_val} x 2^{e_val} = {result}")
    
    if not normalised:
        new_m, new_e, steps = normalise(mantissa, exponent)
        new_m_str = ''.join(map(str, new_m))
        new_m_val = mantissa_value(new_m)
        new_result = new_m_val * (2 ** new_e)
        print(f"  规格化过程:")
        for step in steps:
            print(step)
        print(f"  结果: [{new_m_str} | exp={new_e}] = {new_m_val} x 2^{new_e} = {new_result}")
    print()

---

## 三、精度、范围、上溢和下溢

### 精度 Precision vs 范围 Range

在固定总位数下，尾数和阶码之间存在**权衡 (trade-off)**：

| 增加尾数位 | 增加阶码位 |
|:---|:---|
| 更高的精度 | 更大的范围 |
| 能表示更多有效数字 | 能表示更大/更小的数 |
| 范围缩小 | 精度降低 |

### 上溢 Overflow 和下溢 Underflow

- **上溢 Overflow**：运算结果**太大**，超出了能表示的最大值
- **下溢 Underflow**：运算结果**太接近0**，小于能表示的最小正数（非零）

In [ ]:
# === 精度与范围的权衡 Precision vs Range Trade-off ===

def get_range_and_precision(total_bits, mantissa_bits):
    """计算给定分配下的范围和精度"""
    exponent_bits = total_bits - mantissa_bits
    
    # 最大正数（尾数全1除符号位，阶码最大正值）
    max_mantissa = sum(2**(-i) for i in range(1, mantissa_bits))
    max_exponent = 2**(exponent_bits - 1) - 1
    max_value = max_mantissa * (2 ** max_exponent)
    
    # 最小正数（尾数=0.1000..., 阶码最小负值）
    min_mantissa = 0.5
    min_exponent = -(2**(exponent_bits - 1))
    min_value = min_mantissa * (2 ** min_exponent)
    
    # 精度（最小的可分辨差异）
    precision = 2**(-(mantissa_bits - 1))
    
    return max_value, min_value, precision

total = 8
print(f"=== {total}-bit 浮点数：尾数与阶码的分配 ===")
print(f"{'Mantissa bits':<16} {'Exponent bits':<16} {'Max Value':<16} {'Min Positive':<20} {'Precision':<12}")
print("-" * 80)

configs = []
for m_bits in range(3, total - 1):
    e_bits = total - m_bits
    max_v, min_v, prec = get_range_and_precision(total, m_bits)
    configs.append((m_bits, e_bits, max_v, min_v, prec))
    print(f"{m_bits:<16} {e_bits:<16} {max_v:<16.4f} {min_v:<20.8f} {prec:<12.6f}")

print("\n观察：")
print("  - 增加尾数位 -> 精度提高（precision变小）但范围缩小")
print("  - 增加阶码位 -> 范围增大但精度降低")

In [ ]:
# === 可视化：精度与范围的权衡 ===

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

m_bits_list = [c[0] for c in configs]
max_values = [c[2] for c in configs]
precisions = [c[4] for c in configs]

# 范围图
ax1.bar(m_bits_list, max_values, color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db', '#9b59b6'][:len(configs)],
        edgecolor='black', linewidth=1.5)
ax1.set_xlabel('Mantissa Bits', fontsize=12)
ax1.set_ylabel('Maximum Value', fontsize=12)
ax1.set_title('Range: Max representable value\n\u8303\u56f4\uff1a\u53ef\u8868\u793a\u7684\u6700\u5927\u503c', 
              fontsize=13, fontweight='bold')
ax1.set_xticks(m_bits_list)
ax1.set_xticklabels([f'{m}M/{8-m}E' for m in m_bits_list])
ax1.grid(axis='y', alpha=0.3)

# 精度图
ax2.bar(m_bits_list, precisions, color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db', '#9b59b6'][:len(configs)],
        edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Mantissa Bits', fontsize=12)
ax2.set_ylabel('Precision (smaller = better)', fontsize=12)
ax2.set_title('Precision: Smallest distinguishable difference\n\u7cbe\u5ea6\uff1a\u6700\u5c0f\u53ef\u5206\u8fa8\u5dee\u5f02', 
              fontsize=13, fontweight='bold')
ax2.set_xticks(m_bits_list)
ax2.set_xticklabels([f'{m}M/{8-m}E' for m in m_bits_list])
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Precision vs Range Trade-off in 8-bit Floating Point\n\u7cbe\u5ea6\u4e0e\u8303\u56f4\u7684\u6743\u8861', 
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

---

## 四、浮点数运算 Floating Point Arithmetic

### 加法/减法步骤

1. **对齐阶码 Align exponents**：将较小阶码的数的尾数**右移**，使两个阶码相等
2. **尾数运算 Add/Subtract mantissas**：对齐后，直接加/减尾数
3. **规格化 Normalise**：如果结果不是规格化的，进行调整
4. **检查溢出 Check overflow/underflow**

### 为什么要对齐阶码？

就像十进制中，你不能直接把 3.5 x 10^2 和 2.1 x 10^5 的尾数相加。必须先把它们统一到同一个阶码。

In [ ]:
# === 浮点数加法逐步演示 ===

def float_addition_demo(m1, e1, m2, e2):
    """浮点数加法的逐步过程（十进制简化版）"""
    print(f"=== 浮点数加法 Floating Point Addition ===")
    print(f"\n数1: {m1} x 2^{e1} = {m1 * 2**e1}")
    print(f"数2: {m2} x 2^{e2} = {m2 * 2**e2}")
    
    # Step 1: 对齐阶码
    print(f"\nStep 1: 对齐阶码 Align Exponents")
    if e1 > e2:
        diff = e1 - e2
        print(f"  阶码差 = {e1} - {e2} = {diff}")
        print(f"  将数2的尾数右移{diff}位（阶码+{diff}）")
        m2_aligned = m2 / (2**diff)
        e_common = e1
        print(f"  数2变为: {m2_aligned} x 2^{e_common}")
        m1_aligned = m1
    elif e2 > e1:
        diff = e2 - e1
        print(f"  阶码差 = {e2} - {e1} = {diff}")
        print(f"  将数1的尾数右移{diff}位（阶码+{diff}）")
        m1_aligned = m1 / (2**diff)
        e_common = e2
        print(f"  数1变为: {m1_aligned} x 2^{e_common}")
        m2_aligned = m2
    else:
        print(f"  阶码已相同，无需对齐")
        m1_aligned = m1
        m2_aligned = m2
        e_common = e1
    
    # Step 2: 尾数相加
    print(f"\nStep 2: 尾数相加 Add Mantissas")
    m_result = m1_aligned + m2_aligned
    print(f"  {m1_aligned} + {m2_aligned} = {m_result}")
    print(f"  结果: {m_result} x 2^{e_common}")
    
    # Step 3: 规格化
    print(f"\nStep 3: 规格化 Normalise")
    e_result = e_common
    m_norm = m_result
    if m_norm != 0:
        while abs(m_norm) >= 1:
            m_norm /= 2
            e_result += 1
            print(f"  尾数右移: {m_norm} x 2^{e_result}")
        while 0 < abs(m_norm) < 0.5:
            m_norm *= 2
            e_result -= 1
            print(f"  尾数左移: {m_norm} x 2^{e_result}")
    
    final = m_norm * (2**e_result)
    print(f"\n最终结果: {m_norm} x 2^{e_result} = {final}")
    print(f"验证: {m1 * 2**e1} + {m2 * 2**e2} = {m1 * 2**e1 + m2 * 2**e2}")

# 例题1
float_addition_demo(0.75, 2, 0.625, 4)  # 3 + 10 = 13

In [ ]:
# 例题2：负数加法
print()
float_addition_demo(0.5, 3, -0.75, 2)  # 4 + (-3) = 1

---

## 五、经典问题：为什么 0.1 + 0.2 != 0.3?

这是计算机科学中最著名的"陷阱"之一，也是面试和考试的高频题。

### 根本原因

十进制的 0.1 在二进制中是一个**无限循环小数**，就像十进制中的 1/3 = 0.3333... 一样无法精确表示。

$$0.1_{10} = 0.0001100110011001100110011..._{2}$$

由于尾数位数有限，必须在某处**截断（truncate）**，这就引入了**舍入误差（rounding error）**。

In [ ]:
# === 0.1 + 0.2 != 0.3 的完整解释 ===

print("=== 为什么 0.1 + 0.2 != 0.3? ===")
print()

# 展示Python中的结果
print(f"Python计算: 0.1 + 0.2 = {0.1 + 0.2}")
print(f"0.1 + 0.2 == 0.3? {0.1 + 0.2 == 0.3}")
print(f"差值: {0.1 + 0.2 - 0.3}")
print()

# 展示精确存储的值
from decimal import Decimal
print("=== 计算机实际存储的值（精确到55位小数）===")
print(f"0.1 实际存储为: {Decimal(0.1)}")
print(f"0.2 实际存储为: {Decimal(0.2)}")
print(f"0.3 实际存储为: {Decimal(0.3)}")
print(f"0.1 + 0.2 =     {Decimal(0.1) + Decimal(0.2)}")
print()

# 展示二进制表示
def decimal_to_binary_fraction(n, bits=20):
    """将小数转换为二进制小数表示"""
    result = "0."
    for _ in range(bits):
        n *= 2
        if n >= 1:
            result += "1"
            n -= 1
        else:
            result += "0"
    return result + "..."

print("=== 二进制表示 ===")
print(f"0.1 = {decimal_to_binary_fraction(0.1, 24)} (无限循环!)")
print(f"0.2 = {decimal_to_binary_fraction(0.2, 24)} (无限循环!)")
print(f"0.3 = {decimal_to_binary_fraction(0.3, 24)} (无限循环!)")
print()
print("关键：0.1 和 0.2 在二进制中都是无限循环小数")
print("      截断后的误差在相加时累积，导致结果不等于0.3")

In [ ]:
# === 可视化：二进制转换过程 ===

def show_conversion_process(decimal_val, name, ax, max_steps=12):
    """可视化十进制小数转二进制的乘2取整过程"""
    n = decimal_val
    bits = []
    values = []
    
    for step in range(max_steps):
        n *= 2
        if n >= 1:
            bits.append(1)
            values.append(n)
            n -= 1
        else:
            bits.append(0)
            values.append(n)
    
    steps = range(1, max_steps + 1)
    colors = ['#e74c3c' if b == 1 else '#3498db' for b in bits]
    
    ax.bar(steps, values, color=colors, edgecolor='black', linewidth=0.5, alpha=0.8)
    for i, (b, v) in enumerate(zip(bits, values)):
        ax.text(i + 1, v + 0.05, str(b), ha='center', fontsize=10, fontweight='bold')
    
    binary_str = '0.' + ''.join(map(str, bits))
    ax.set_title(f'{name} = {decimal_val}\nBinary: {binary_str}...', fontsize=11, fontweight='bold')
    ax.set_xlabel('Step (multiply by 2)', fontsize=9)
    ax.set_ylabel('Value after x2', fontsize=9)
    ax.set_ylim(0, 2.2)
    ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
    ax.grid(axis='y', alpha=0.3)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
show_conversion_process(0.1, '0.1', axes[0])
show_conversion_process(0.2, '0.2', axes[1])
show_conversion_process(0.5, '0.5 (exact!)', axes[2])

plt.suptitle('Decimal to Binary Conversion Process (Multiply-by-2 Method)\n\u5341\u8fdb\u5236\u8f6c\u4e8c\u8fdb\u5236\u7684\u4e58\u4ee5\u0032\u53d6\u6574\u6cd5', 
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print("观察：0.1 和 0.2 的转换过程中数值不断循环，永远到不了0")
print("     但 0.5 只需一步就精确完成！因为 0.5 = 2^(-1) = 0.1 (binary)")

In [ ]:
# === 可视化：浮点数在数轴上的分布 ===
# 这解释了为什么靠近0的浮点数更密集

def generate_mini_floats(mantissa_bits=4, exponent_bits=3):
    """生成所有可能的迷你浮点数值"""
    values = set()
    
    for m_int in range(2**mantissa_bits):
        for e_int in range(2**exponent_bits):
            # 转换为位列表
            m_bits = [(m_int >> (mantissa_bits - 1 - i)) & 1 for i in range(mantissa_bits)]
            e_bits = [(e_int >> (exponent_bits - 1 - i)) & 1 for i in range(exponent_bits)]
            
            m_val = mantissa_value(m_bits)
            e_val = twos_complement_value(e_bits)
            result = m_val * (2 ** e_val)
            
            # 只取规格化的数
            if m_bits[0] != m_bits[1] if len(m_bits) >= 2 else True:
                values.add(result)
    
    return sorted(values)

values = generate_mini_floats(4, 3)

fig, axes = plt.subplots(2, 1, figsize=(16, 7))

# 完整数轴
ax = axes[0]
ax.scatter(values, [0]*len(values), s=50, c='#e74c3c', zorder=5, alpha=0.7)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_xlim(min(values) - 0.5, max(values) + 0.5)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_title('All Representable Floating Point Numbers (4-bit mantissa, 3-bit exponent)\n\u6240\u6709\u53ef\u8868\u793a\u7684\u6d6e\u70b9\u6570', 
             fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# 靠近0的放大图
ax = axes[1]
close_to_zero = [v for v in values if -1 <= v <= 1]
ax.scatter(close_to_zero, [0]*len(close_to_zero), s=80, c='#3498db', zorder=5, alpha=0.8)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_title('Zoomed in: Numbers near 0 are MORE DENSELY packed!\n\u653e\u5927\uff1a\u9760\u8fd1\u0030\u7684\u6570\u66f4\u5bc6\u96c6\uff01', 
             fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# 标注
for v in close_to_zero:
    if v != 0:
        ax.annotate(f'{v:.3f}', xy=(v, 0), xytext=(v, 0.25 if close_to_zero.index(v) % 2 == 0 else -0.25),
                    fontsize=7, ha='center', rotation=45,
                    arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

plt.tight_layout()
plt.show()

print(f"共有 {len(values)} 个可表示的值")
print(f"范围: [{min(values)}, {max(values)}]")
print()
print("关键观察 Key Observation:")
print("  靠近0的浮点数分布密集，远离0的浮点数分布稀疏")
print("  原因：阶码越大，相邻浮点数之间的间隔越大")
print("  这意味着：大数的精度低于小数的精度！")

In [ ]:
# === 可视化：浮点数间距随大小变化 ===

# 使用Python的float展示间距变化
def get_float_spacing(x):
    """获取浮点数 x 附近的间距"""
    return np.spacing(x)

x_values = np.logspace(-10, 15, 100)
spacings = [np.spacing(x) for x in x_values]

fig, ax = plt.subplots(figsize=(12, 6))
ax.loglog(x_values, spacings, 'o-', color='#e74c3c', linewidth=2, markersize=4)

ax.set_xlabel('Number value (log scale)', fontsize=12)
ax.set_ylabel('Spacing to next float (log scale)', fontsize=12)
ax.set_title('Float Spacing Increases with Magnitude\n\u6d6e\u70b9\u6570\u95f4\u8ddd\u968f\u6570\u503c\u589e\u5927\u800c\u589e\u5927', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# 标注
ax.annotate('Near 0: very precise\n\u9760\u8fd10\uff1a\u975e\u5e38\u7cbe\u786e', 
            xy=(1e-8, np.spacing(1e-8)), fontsize=11, fontweight='bold',
            color='#27ae60',
            xytext=(1e-4, np.spacing(1e-8)*1e5),
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))

ax.annotate('Large numbers: imprecise\n\u5927\u6570\uff1a\u7cbe\u5ea6\u4f4e', 
            xy=(1e12, np.spacing(1e12)), fontsize=11, fontweight='bold',
            color='#e74c3c',
            xytext=(1e8, np.spacing(1e12)*1e3),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))

plt.tight_layout()
plt.show()

print("实际例子：")
print(f"  1.0 附近的精度: {np.spacing(1.0):.2e}")
print(f"  1000000.0 附近的精度: {np.spacing(1000000.0):.2e}")
print(f"  1e15 附近的精度: {np.spacing(1e15):.2e}")
print(f"\n  这意味着 1e15 + 0.5 可能等于 1e15 (精度不够，0.5被丢弃)!")
print(f"  验证: {1e15 + 0.5 == 1e15}")

---

## 六、IEEE 754 标准简介（拓展知识）

IEEE 754 是现代计算机中浮点数表示的**国际标准**。

### 两种常用格式

| 格式 | 总位数 | 符号 | 阶码 | 尾数 | 约等于十进制精度 |
|:---|:---|:---|:---|:---|:---|
| **单精度 (float)** | 32 | 1 | 8 | 23 | 7位有效数字 |
| **双精度 (double)** | 64 | 1 | 11 | 52 | 15-16位有效数字 |

### 与Cambridge考试格式的区别

- IEEE 754 用**移码 (biased exponent)** 而非二进制补码表示阶码
- IEEE 754 尾数有一个**隐含的1**（leading 1），节省1位
- 有特殊值：+/-Infinity、NaN (Not a Number)

In [ ]:
# === IEEE 754 内部结构探秘 ===

def show_ieee754(value):
    """展示一个浮点数的IEEE 754二进制表示"""
    # 将float转为bytes，再转为二进制字符串
    packed = struct.pack('>d', value)  # 双精度，大端序
    binary = ''.join(f'{byte:08b}' for byte in packed)
    
    sign = binary[0]
    exponent = binary[1:12]
    mantissa = binary[12:]
    
    sign_val = int(sign)
    exp_val = int(exponent, 2) - 1023  # IEEE 754双精度偏移量
    
    print(f"  Value: {value}")
    print(f"  Sign:     {sign} ({'negative' if sign_val else 'positive'})")
    print(f"  Exponent: {exponent} (biased: {int(exponent, 2)}, actual: {exp_val})")
    print(f"  Mantissa: {mantissa[:20]}... (1.{mantissa[:20]}... in binary)")

print("=== IEEE 754 Double Precision (64-bit) ===")
print()
for val in [1.0, -1.0, 0.1, 0.5, 3.14, 100.0]:
    show_ieee754(val)
    print()

# 特殊值
print("=== 特殊值 Special Values ===")
print(f"  float('inf'):  {float('inf')}")
print(f"  float('-inf'): {float('-inf')}")
print(f"  float('nan'):  {float('nan')}")
print(f"  float('nan') == float('nan'): {float('nan') == float('nan')} (NaN不等于任何值，包括自己!)")

In [ ]:
# === 浮点数陷阱总结与安全比较方法 ===

print("=== 浮点数常见陷阱 Common Pitfalls ===")
print()

# 陷阱1：相等比较
print("陷阱1: 直接比较相等")
print(f"  0.1 + 0.2 == 0.3: {0.1 + 0.2 == 0.3}  <-- 错误!")
print(f"  正确做法: abs((0.1+0.2) - 0.3) < 1e-9: {abs((0.1+0.2) - 0.3) < 1e-9}")
print(f"  或使用 math.isclose: ", end="")
import math
print(f"{math.isclose(0.1 + 0.2, 0.3)}")
print()

# 陷阱2：大数吞小数
print("陷阱2: 大数+小数 = 大数（小数被吞）")
big = 1e16
small = 1.0
print(f"  {big} + {small} == {big}: {big + small == big}")
print(f"  因为 1.0 在 10^16 的精度范围之外")
print()

# 陷阱3：累积误差
print("陷阱3: 累积误差")
total = 0.0
for _ in range(10):
    total += 0.1
print(f"  0.1 加 10 次 = {total} (不是 1.0!)")
print(f"  误差: {total - 1.0:.2e}")
print()

# 陷阱4：减法消去
print("陷阱4: 减法消去 (Catastrophic Cancellation)")
a = 1.0000000000000002
b = 1.0000000000000001
print(f"  a = {a}")
print(f"  b = {b}")
print(f"  a - b = {a - b} (应该是 1e-16，但精度已丢失)")

---

## 练习题 Exercises

### 练习1：转换题

使用 **6位尾数 + 4位阶码** 的格式，将以下浮点数转换为十进制：

1. `010100 | 0011`
2. `011000 | 0001`
3. `101100 | 0100`
4. `100000 | 1110`

### 练习2：规格化题

判断以下浮点数是否已规格化，如果未规格化，请规格化：

1. Mantissa: `001100`, Exponent: `011`
2. Mantissa: `010100`, Exponent: `010`
3. Mantissa: `110010`, Exponent: `100`
4. Mantissa: `101000`, Exponent: `011`

### 练习3：运算题

使用 5位尾数 + 3位阶码 格式，计算：
- `01100|010` + `01000|011` = ?

### 练习4：编程题

In [ ]:
# 练习4a：编写函数 decimal_to_float(value, m_bits, e_bits)
# 将一个十进制数转换为指定格式的浮点数位模式
# 返回 (mantissa_bits, exponent_bits)

# 你的代码：


In [ ]:
# 练习4b：实现浮点数的二进制加法器
# 输入两个浮点数的位模式，输出它们的和的位模式
# 要求包含完整的对齐、相加、规格化步骤

# 你的代码：


In [ ]:
# 练习4c：可视化实验
# 绘制一个图表，展示不断将 0.1 累加的误差增长曲线
# x轴：累加次数（1到1000）
# y轴：实际累加值 vs 理论值（n * 0.1）的误差

# 你的代码：


### 练习5：考试题风格

**Question 1** (Cambridge style)

A floating-point number is stored using 8 bits: 5 bits for the mantissa and 3 bits for the exponent. Both use two's complement representation. The mantissa represents a fractional value.

(a) Convert the floating-point number `01011 | 011` to denary. Show your working. [4]

(b) The number `00101 | 010` is not normalised. Explain why, and normalise it. [4]

(c) Explain the trade-off between increasing the number of mantissa bits and increasing the number of exponent bits. [3]

(d) Explain why 0.1 cannot be represented exactly in binary floating point. [3]

(e) Describe one situation where floating-point overflow could occur. [2]

---

### 本节小结 Summary

- 浮点数 = **尾数** x 2^**阶码**，分别用二进制补码表示
- **规格化**确保最高精度：正数以0.1开头，负数以1.0开头
- 尾数位数决定**精度**，阶码位数决定**范围**，两者存在权衡
- 浮点数在数轴上**分布不均匀**：靠近0更密集，远离0更稀疏
- **0.1+0.2 != 0.3** 是因为0.1在二进制中是无限循环小数
- 比较浮点数应使用**误差范围**而非直接相等比较